# XGBoost đối chiếu LightGBM tại cutoff 50% và 75%

**Câu hỏi:** giữ nguyên dữ liệu, cách chia sinh viên và ngân sách HCO/VNLS,
thay LightGBM bằng XGBoost có cải thiện dự báo nguy cơ không?

Model app vẫn là **HCO–VNLS + LightGBM tại cutoff 75%**. Đây là đối chứng,
không ghi đè model đó. Code học, chọn biến, đo chỉ số và vẽ hình nằm trong các cell.
Đọc: giao thức → kiểm tra dữ liệu → bộ học → tìm biến → kết quả → nhận xét.

**Full features** là đủ 38 biến tại cutoff tương ứng, không phải dữ liệu toàn kỳ 100%.
Chỉ đánh giá validation; không fit, dự đoán hoặc dò tham số trên test.
Giảm số biến không mặc định tăng Accuracy hoặc giảm thời gian.

In [ ]:
from pathlib import Path
import hashlib, importlib.metadata, json, platform, sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.model_selection import GroupKFold
from sklearn.metrics import average_precision_score
from lightgbm import LGBMClassifier

ROOT = next((directory for directory in [Path.cwd(), *Path.cwd().parents]
             if (directory / 'research_code').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from packages/ml or its notebooks directory.")
sys.path.insert(0, str(ROOT))
from shared_code.thuc_nghiem import nap_du_lieu, chon_dac_trung, KET_QUA, HINH
from research_code.models import ModelConfig, make_pipeline
from research_code.evaluation import metric_table
from research_code.optimizers import run_search_budgeted, fitness

plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10})
VERSIONS = {package: importlib.metadata.version(package)
            for package in ('numpy', 'pandas', 'scikit-learn', 'lightgbm', 'catboost', 'xgboost')}
print('Python:', platform.python_version())
display(pd.DataFrame(VERSIONS.items(), columns=['Thư viện', 'Phiên bản']))

## 1. Giao thức cố định

Cutoff 50%/75% là thời điểm quan sát trên độ dài học phần, **không lấy 50%/75% số dòng**.
Giữ cột split đã chia theo sinh viên ở seed 41: khoảng 70%/15%/15% số sinh viên.
Tỷ lệ lượt học có thể khác vì mỗi sinh viên có nhiều lượt đăng ký.

HCO, VNLS và HCO–VNLS chỉ xem **train**, dùng GroupKFold 3 phần.
Mỗi phương pháp có tối đa 8 mask mới; cache không tiêu thêm ngân sách.

$$F(m)=0.85\,\overline{AP}_{train\text{-}CV}(m)+0.15\left(1-\frac{|m|}{38}\right)$$

m là mask bật/tắt 38 biến; |m| là số biến bật. Chọn theo fitness, không theo Accuracy validation.
Ngưỡng phân loại cố định 0,5; không tối ưu ngưỡng. VNLS hiện tại chủ yếu lật một bit,
không gọi đây là triển khai đầy đủ mọi lân cận trong lý thuyết.

LightGBM được fit lại trên train với mask cũ đã chốt, không tìm mask lại.
Không trộn thời gian tìm mask cũ với phép đo mới. Cùng 80 cây không đồng nghĩa
cùng độ phức tạp, công bằng tuyệt đối hoặc đã tối ưu tham số từng bộ học.

In [ ]:
MODEL_NAME = 'XGBoost'
SLUG = 'xgboost'
SEED, BUDGET, POPULATION, ALPHA, THRESHOLD = 41, 8, 3, 0.85, 0.5
METHODS = ['Full features', 'HCO', 'VNLS', 'HCO-VNLS']
LIGHTGBM_PARAMS = dict(n_estimators=80, learning_rate=.055, num_leaves=15,
                      min_child_samples=35, verbosity=-1, random_state=SEED, n_jobs=4)
RESULT_PATH = KET_QUA / f'so_sanh_{SLUG}_lightgbm_50_75.csv'
TRACE_PATH = KET_QUA / f'tim_mask_{SLUG}_50_75.csv'
META_PATH = KET_QUA / f'giao_thuc_{SLUG}_50_75.json'
FROZEN_PATH = KET_QUA / 'hco_vnls_lightgbm_validation.csv'
frozen = pd.read_csv(FROZEN_PATH)
print('Seed:', SEED, '| mask mỗi phương pháp:', BUDGET, '| ngưỡng:', THRESHOLD)

## 2. Kiểm tra dữ liệu và rò rỉ

Nạp hai CSV cutoff từ quy trình hiện tại; SHA-256 ghi dấu vết dữ liệu.
Chỉ hiển thị thống kê tổng hợp, không đưa mã sinh viên vào output.
Chín cột định danh/nhãn/cấu hình không đi vào bộ chọn biến.
Kiểm tra nhóm sinh viên không giao nhau và cùng phép chia ở hai cutoff.
Chỉ train/validation được đưa vào thí nghiệm; test chỉ đếm số dòng, không đánh giá.
Không fit bộ điền thiếu trên toàn bảng trước khi chia train.

In [ ]:
datasets, audit_rows, data_hashes, student_assignments = {}, [], {}, {}
excluded_by_cutoff = {}
for cutoff in (.50, .75):
    frame = nap_du_lieu(cutoff)
    columns = chon_dac_trung(frame)
    assert len(columns) == 38
    assert not frame.duplicated(['id_student', 'code_module', 'code_presentation']).any()
    assert set(frame.split) == {'train', 'validation', 'test'}
    assert frame.groupby('id_student').split.nunique().max() == 1
    for student, split in frame[['id_student', 'split']].drop_duplicates().itertuples(index=False):
        assert student not in student_assignments or student_assignments[student] == split
        student_assignments[student] = split
    train = frame.loc[frame.split.eq('train')].copy()
    valid = frame.loc[frame.split.eq('validation')].copy()
    assert not set(train.id_student).intersection(valid.id_student)
    assert set(train.risk) == {0, 1} and set(valid.risk) == {0, 1}
    datasets[cutoff] = (train, valid, columns)
    excluded_by_cutoff[str(cutoff)] = [column for column in frame if column not in columns]
    source = KET_QUA / f'du_lieu_cutoff_{round(cutoff * 100):02d}.csv'
    data_hashes[str(cutoff)] = hashlib.sha256(source.read_bytes()).hexdigest()
    audit_rows.append(dict(cutoff=cutoff, rows=len(frame), raw_columns=frame.shape[1],
                           train_rows=len(train), validation_rows=len(valid),
                           test_rows=int(frame.split.eq('test').sum()), features=len(columns),
                           train_students=train.id_student.nunique(),
                           validation_students=valid.id_student.nunique(),
                           risk_rate_train=train.risk.mean(), risk_rate_validation=valid.risk.mean()))
audit = pd.DataFrame(audit_rows)
display(audit)
print('Cột loại khỏi đầu vào:', excluded_by_cutoff['0.75'])

## 3. XGBoost và tiền xử lý

Cấu hình khởi đầu: 80 cây, max_depth 4, learning rate 0,055, histogram CPU,
L2 bằng 1, không lấy mẫu dòng/cột ngẫu nhiên. **Chưa phải cấu hình tối ưu**.

Dùng cùng tiền xử lý như LightGBM: median số, mode chữ rồi One-Hot,
category mới được bỏ qua khi transform. Các thống kê chỉ fit trên train
của từng fold. Không dùng early stopping dựa trên validation.
Cùng số cây không đồng nghĩa cùng cấu trúc cây hoặc chi phí.

In [ ]:
from xgboost import XGBClassifier
MODEL_PARAMS = dict(n_estimators=80, max_depth=4, learning_rate=.055,
                    min_child_weight=1, reg_lambda=1.0, reg_alpha=0.0,
                    subsample=1.0, colsample_bytree=1.0, tree_method='hist',
                    objective='binary:logistic', eval_metric='logloss',
                    random_state=SEED, n_jobs=4, verbosity=0, device='cpu')

def measure_model(model_name, train_x, train_y, valid_x):
    started = time.perf_counter()
    estimator = (XGBClassifier(**MODEL_PARAMS) if model_name == 'XGBoost'
                 else LGBMClassifier(**LIGHTGBM_PARAMS))
    model = make_pipeline(train_x, ModelConfig(model_name, estimator))
    model.fit(train_x, train_y)
    fit_seconds = time.perf_counter() - started
    started = time.perf_counter()
    probability = model.predict_proba(valid_x)[:, 1]
    predict_seconds = time.perf_counter() - started
    encoded = len(model.named_steps['preprocess'].get_feature_names_out())
    return probability, fit_seconds, predict_seconds, encoded

print('Cấu hình XGBoost:', MODEL_PARAMS)

## 4. Một mask được chấm như thế nào?

Chọn các cột bật → chia train thành 3 fold theo sinh viên → fit 3 lần → lấy trung bình AP.
Thay AP vào fitness, có phần thưởng giảm biến. Cache tránh fit lại mask trùng.
std_ap_cv là độ phân tán AP **giữa ba fold của một mask** (ddof=0), không phải qua nhiều seed
và không phải khoảng tin cậy. Mỗi điểm hội tụ là một mask mới đã chấm, không phải epoch.

In [ ]:
class ComparisonMaskObjective:
    def __init__(self, frame, labels, groups):
        self.frame = frame
        self.labels, self.groups = np.asarray(labels), np.asarray(groups)
        self.cache, self.fold_stats, self.records = {}, {}, []
        self.calls, self.cache_hits = 0, 0

    def __call__(self, mask):
        key = tuple(np.asarray(mask, dtype=np.int8))
        if key in self.cache:
            self.cache_hits += 1
            return self.cache[key]
        if self.calls >= BUDGET:
            raise RuntimeError('Vượt ngân sách mask mới; cần kiểm tra tìm kiếm.')
        selected = self.frame.columns[np.flatnonzero(mask)].tolist()
        if not selected:
            return 0.0, 0.0
        fold_ap = []
        for fold_train, fold_valid in GroupKFold(n_splits=3).split(self.frame, self.labels, self.groups):
            assert not set(self.groups[fold_train]).intersection(self.groups[fold_valid])
            probability, _, _, _ = measure_model(MODEL_NAME, self.frame.iloc[fold_train][selected],
                                                 self.labels[fold_train], self.frame.iloc[fold_valid][selected])
            fold_ap.append(float(average_precision_score(self.labels[fold_valid], probability)))
        mean_ap = float(np.mean(fold_ap))
        value = fitness(mean_ap, len(selected), self.frame.shape[1], ALPHA)
        self.cache[key], self.fold_stats[key] = (value, mean_ap), fold_ap
        self.calls += 1
        best_fitness = max([record['best_fitness'] for record in self.records] + [value])
        self.records.append(dict(evaluation=self.calls, mask=''.join(map(str, key)), features=len(selected),
                                 fitness_cv=value, mean_ap_cv=mean_ap, std_ap_cv=float(np.std(fold_ap)),
                                 best_fitness=best_fitness))
        return value, mean_ap

## 5. Chạy bốn trường hợp của mỗi bộ học

Mỗi cutoff: LightGBM với bốn mask đã chốt; bộ học mới với đủ biến/HCO/VNLS/HCO–VNLS.
Mỗi tìm kiếm có cache riêng. Model cuối mỗi mask chỉ fit train, đánh giá trên cùng validation.

fit_seconds gồm tiền xử lý + fit; predict_seconds gồm transform + dự đoán validation.
search_seconds là thời gian **tìm mask**, tách khỏi fit cuối. LightGBM không tìm lại mask nên
thời gian tìm mới là NaN, không ghi 0 như thể miễn phí. Mỗi thời gian đo một lần;
chưa đủ khẳng định tiết kiệm tài nguyên hoặc tốc độ ổn định.

In [ ]:
rows, traces = [], []
for cutoff, (train, valid, columns) in datasets.items():
    print(f'Cutoff {cutoff:.0%}: train {len(train):,}, validation {len(valid):,}')
    reference = frozen.loc[np.isclose(frozen.cutoff, cutoff)]
    assert set(reference.method) == set(METHODS) and len(reference) == 4
    for method in METHODS:
        source = reference.loc[reference.method.eq(method)].iloc[0]
        selected = columns if method == 'Full features' else source.selected_columns.split('|')
        assert set(selected).issubset(columns) and len(selected) == source.features
        probability, fit_time, predict_time, encoded = measure_model('LightGBM', train[selected], train.risk, valid[selected])
        metrics = metric_table(valid.risk, probability, THRESHOLD)
        rows.append(dict(model='LightGBM', cutoff=cutoff, method=method, mask_source='frozen_lightgbm_05b',
                         features=len(selected), reduction_pct=100*(1-len(selected)/len(columns)), encoded_features=encoded,
                         fitness_cv=source.fitness_cv, mean_ap_cv=np.nan, std_ap_cv=np.nan, objective_calls=0,
                         search_seconds=np.nan, fit_seconds=fit_time, predict_seconds=predict_time,
                         historical_ap=source.average_precision_risk,
                         ap_delta_vs_historical=metrics['average_precision_risk']-source.average_precision_risk,
                         selected_columns='|'.join(selected), **metrics))
    for method in METHODS:
        search_time, mean_ap, std_ap, fit_value, calls = 0.0, np.nan, np.nan, np.nan, 0
        selected = columns
        if method != 'Full features':
            objective = ComparisonMaskObjective(train[columns], train.risk, train.id_student)
            search = run_search_budgeted(method, objective, seed=SEED, budget=BUDGET, population=POPULATION)
            selected = search['selected_columns']
            search_time, fit_value, mean_ap = search['wall_seconds'], search['fitness_cv'], search['average_precision_cv']
            calls = search['objective_calls']
            key = tuple(int(column in selected) for column in columns)
            std_ap = float(np.std(objective.fold_stats[key]))
            assert calls == BUDGET
            traces.extend(dict(cutoff=cutoff, method=method, **record) for record in objective.records)
        probability, fit_time, predict_time, encoded = measure_model(MODEL_NAME, train[selected], train.risk, valid[selected])
        metrics = metric_table(valid.risk, probability, THRESHOLD)
        rows.append(dict(model=MODEL_NAME, cutoff=cutoff, method=method,
                         mask_source='all_features' if method == 'Full features' else 'train_group_cv',
                         features=len(selected), reduction_pct=100*(1-len(selected)/len(columns)), encoded_features=encoded,
                         fitness_cv=fit_value, mean_ap_cv=mean_ap, std_ap_cv=std_ap, objective_calls=calls,
                         search_seconds=search_time, fit_seconds=fit_time, predict_seconds=predict_time,
                         historical_ap=np.nan, ap_delta_vs_historical=np.nan, selected_columns='|'.join(selected), **metrics))
        print(f'  {method:13s}: {len(selected):2d} biến; Acc {metrics["accuracy"]:.4f}; '
              f'AP {metrics["average_precision_risk"]:.4f}; search {search_time:.1f}s')
result, trajectory = pd.DataFrame(rows), pd.DataFrame(traces)
assert len(result) == 16 and not result.duplicated(['model', 'cutoff', 'method']).any()
reference_drift = float(result.loc[result.model.eq('LightGBM'), 'ap_delta_vs_historical'].abs().max())
print('Chênh AP lớn nhất khi tái lập LightGBM:', reference_drift)
display(result[['cutoff', 'model', 'method', 'features', 'accuracy', 'precision_risk', 'recall_risk',
                'f1_risk', 'weighted_f1', 'average_precision_risk', 'roc_auc', 'brier_score',
                'fit_seconds', 'predict_seconds', 'search_seconds']].round(5))

## 6. Chỉ số và danh sách biến được chọn

Accuracy: tỷ lệ đúng toàn bộ; Precision risk: tỷ lệ cảnh báo đúng;
Recall risk: phát hiện bao nhiêu ca nguy cơ thực; F1 risk: cân bằng Precision/Recall lớp nguy cơ.
Weighted F1: trung bình F1 hai lớp có trọng số. AP: chất lượng xếp hạng qua nhiều ngưỡng.
ROC AUC: xếp ca nguy cơ cao hơn ca không nguy cơ; Brier: sai số xác suất, thấp hơn tốt hơn.

Trước tiên so giảm biến với đủ 38 biến **cùng bộ học và cutoff**. 38 còn 17 là giảm 55,26%
biến gốc, không tự động giảm 55,26% thời gian/dung lượng. Tên biến giữ nguyên để đối chiếu code.

In [ ]:
own = result.loc[result.model.eq(MODEL_NAME)].copy()
baselines = own.loc[own.method.eq('Full features')].set_index('cutoff')
comparison = own.loc[own.method.ne('Full features')].copy()
for metric in ('accuracy', 'average_precision_risk', 'recall_risk', 'f1_risk'):
    comparison[f'delta_{metric}'] = comparison[metric] - comparison.cutoff.map(baselines[metric])
comparison['fit_speed_ratio_full_over_selected'] = comparison.cutoff.map(baselines.fit_seconds) / comparison.fit_seconds
display(comparison[['cutoff', 'method', 'features', 'reduction_pct', 'delta_accuracy',
                    'delta_average_precision_risk', 'delta_recall_risk', 'delta_f1_risk',
                    'fit_speed_ratio_full_over_selected']].round(5))
display(own.loc[own.method.ne('Full features'), ['cutoff', 'method', 'features', 'selected_columns']]
        .style.set_properties(subset=['selected_columns'], **{'white-space': 'normal', 'max-width': '800px'}))

## 7. Vẽ hình và lưu bằng chứng

Hình đầu gồm AP/Recall, Accuracy/F1 và thời gian fit/dự đoán, mỗi cột một cutoff.
Đường nối giúp đọc bốn trường hợp, **không biểu diễn quá trình học**.
Hình thứ hai là fitness tốt nhất train-CV theo số mask mới, không phải Accuracy validation.
Hình lưu ở figures, bảng ở results. Không xuất xác suất cá nhân hoặc model đối chứng.

In [ ]:
figure, axes = plt.subplots(3, 2, figsize=(15, 12), constrained_layout=True)
colors = {'LightGBM': '#336699', MODEL_NAME: '#9a622b'}
for column_index, cutoff in enumerate((.50, .75)):
    subset = result.loc[np.isclose(result.cutoff, cutoff)]
    for name in ('LightGBM', MODEL_NAME):
        ordered = subset.loc[subset.model.eq(name)].set_index('method').loc[METHODS]
        for row_index, metrics in enumerate([('average_precision_risk', 'recall_risk'),
                                             ('accuracy', 'f1_risk'), ('fit_seconds', 'predict_seconds')]):
            for metric, marker in zip(metrics, ['o-', 's--']):
                axes[row_index, column_index].plot(np.arange(4), ordered[metric], marker,
                                                   color=colors[name], label=f'{name}: {metric}')
    for row_index in range(3):
        axis = axes[row_index, column_index]
        axis.set_xticks(np.arange(4), ['Đủ 38 biến', 'HCO', 'VNLS', 'HCO–VNLS'], rotation=15)
        axis.set_title(f'Cutoff {cutoff:.0%}')
        axis.set_ylabel('Điểm (0–1)' if row_index < 2 else 'Thời gian (giây)')
        axis.grid(axis='y', alpha=.25)
        axis.legend(fontsize=7, ncol=1)
        axis.set_ylim(0, 1) if row_index < 2 else axis.set_ylim(bottom=0)
figure.suptitle(f'{MODEL_NAME} và LightGBM — cùng dữ liệu, validation', fontsize=15)
metric_figure = HINH / f'so_sanh_{SLUG}_lightgbm_50_75.png'
figure.savefig(metric_figure, dpi=180)
plt.show()
plt.close(figure)
figure, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
for axis, cutoff in zip(axes, (.50, .75)):
    for method in METHODS[1:]:
        trace = trajectory.loc[np.isclose(trajectory.cutoff, cutoff) & trajectory.method.eq(method)]
        axis.plot(trace.evaluation, trace.best_fitness, 'o-', label=method)
    axis.set(title=f'{MODEL_NAME}: cutoff {cutoff:.0%}', xlabel='Số mask mới đã chấm', ylabel='Fitness tốt nhất (train-CV)')
    axis.set_xticks(range(1, BUDGET + 1))
    axis.grid(alpha=.25)
    axis.legend()
trace_figure = HINH / f'tim_mask_{SLUG}_50_75.png'
figure.savefig(trace_figure, dpi=180)
plt.show()
plt.close(figure)
result.to_csv(RESULT_PATH, index=False, encoding='utf-8-sig')
trajectory.to_csv(TRACE_PATH, index=False, encoding='utf-8-sig')
metadata = dict(model=MODEL_NAME, run_time_local=time.strftime('%Y-%m-%dT%H:%M:%S'),
                python=platform.python_version(), versions=VERSIONS, params=MODEL_PARAMS,
                lightgbm_params=LIGHTGBM_PARAMS, seed=SEED, budget=BUDGET, population=POPULATION,
                alpha=ALPHA, threshold=THRESHOLD, cv='GroupKFold(n_splits=3) inside train',
                split_source='Existing student split column, seed 41', test_evaluated=False,
                production_model_changed=False, data_sha256=data_hashes, excluded_columns=excluded_by_cutoff,
                data_audit=audit.to_dict(orient='records'), lightgbm_max_ap_drift=reference_drift,
                frozen_csv_sha256=hashlib.sha256(FROZEN_PATH.read_bytes()).hexdigest(),
                timing_repeats=1, reference_search_rerun=False,
                missing_policy=('CatBoost numeric NaN; categorical THIEU' if MODEL_NAME == 'CatBoost'
                                else 'train-only median/mode + OneHot'),
                artifacts=[RESULT_PATH.name, TRACE_PATH.name, metric_figure.name, trace_figure.name])
META_PATH.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
print('Đã lưu bảng, hình, giao thức; không thay model app.')

## 8. Nhận xét từ số liệu vừa chạy

Cell lấy số liệu trực tiếp, không viết sẵn rằng bộ học mới thắng.
Đọc mức giảm biến cùng thay đổi AP/Recall/F1. Chênh nhỏ chưa có ý nghĩa thống kê;
một seed và ngân sách 8 chỉ là pilot.

In [ ]:
paragraphs = []
for cutoff in (.50, .75):
    subset = result.loc[np.isclose(result.cutoff, cutoff)]
    full = subset.loc[subset.model.eq(MODEL_NAME) & subset.method.eq('Full features')].iloc[0]
    hybrid = subset.loc[subset.model.eq(MODEL_NAME) & subset.method.eq('HCO-VNLS')].iloc[0]
    reference = subset.loc[subset.model.eq('LightGBM') & subset.method.eq('HCO-VNLS')].iloc[0]
    paragraphs.append(
        f'### Cutoff {cutoff:.0%}\n'
        f'{MODEL_NAME} đủ 38 biến: Accuracy **{full.accuracy:.2%}**, AP **{full.average_precision_risk:.4f}**. '
        f'Hybrid {MODEL_NAME}: **{hybrid.features} biến**, giảm **{hybrid.reduction_pct:.2f}%**, '
        f'Accuracy **{hybrid.accuracy:.2%}**, Precision **{hybrid.precision_risk:.2%}**, '
        f'Recall **{hybrid.recall_risk:.2%}**, F1 risk **{hybrid.f1_risk:.2%}**, AP **{hybrid.average_precision_risk:.4f}**.\n\n'
        f'So hybrid LightGBM: chênh Accuracy **{100*(hybrid.accuracy-reference.accuracy):+.2f} điểm %**, '
        f'AP **{hybrid.average_precision_risk-reference.average_precision_risk:+.4f}**. '
        f'So {MODEL_NAME} đủ biến: AP **{hybrid.average_precision_risk-full.average_precision_risk:+.4f}**, '
        f'Recall **{100*(hybrid.recall_risk-full.recall_risk):+.2f} điểm %**. '
        f'Search **{hybrid.search_seconds:.2f}s**, fit **{hybrid.fit_seconds:.3f}s**, '
        f'dự đoán validation **{hybrid.predict_seconds:.3f}s**.\n')
display(Markdown('\n'.join(paragraphs)))
print('Giữ nguyên HCO–VNLS + LightGBM 75% cho app, không tự thay model triển khai.')

## 9. Giới hạn khi báo cáo

- Chỉ validation trên dataset hiện tại; chưa chứng minh tổng quát sang trường/học kỳ khác.
- Test từng được xem trong lịch sử; không gọi test cũ là holdout mới.
- Một seed, cấu hình cố định, 8 mask; chưa tối ưu hyperparameter hoặc chứng minh tối ưu toàn cục.
- Mỗi bộ học có mask riêng, so hybrid là so pipeline end-to-end,
  không tách riêng tác động của bộ học khỏi tập biến.
- Thời gian đo một lần, cùng máy nhưng nối tiếp; không phải benchmark ổn định.
- Chỉ đánh giá dự báo, không chứng minh gợi ý làm tăng điểm.
- Không xuất dự đoán cá nhân, không ghi đè model/notebook đã chốt, không tự push.

## 10. Bảng gộp ba bộ học

Chạy CatBoost trước để đủ ba bộ học. Lấy LightGBM từ lần đo của notebook này,
không nhân đôi hàng đối chứng. Thời gian đo nối tiếp, không benchmark đồng thời.
Kiểm tra SHA dữ liệu và giao thức khớp trước khi gộp.

In [ ]:
catboost_path = KET_QUA / 'so_sanh_catboost_lightgbm_50_75.csv'
if catboost_path.exists():
    catboost_result = pd.read_csv(catboost_path)
    catboost_meta = json.loads((KET_QUA / 'giao_thuc_catboost_50_75.json').read_text(encoding='utf-8'))
    assert catboost_meta['data_sha256'] == data_hashes
    for name, value in [('seed', SEED), ('budget', BUDGET), ('population', POPULATION),
                        ('alpha', ALPHA), ('threshold', THRESHOLD)]:
        assert catboost_meta[name] == value
    combined = pd.concat([result, catboost_result.loc[catboost_result.model.eq('CatBoost')]], ignore_index=True)
    assert len(combined) == 24 and not combined.duplicated(['model', 'cutoff', 'method']).any()
    combined.to_csv(KET_QUA / 'so_sanh_ba_bo_hoc_50_75.csv', index=False, encoding='utf-8-sig')
    display(combined.loc[combined.method.eq('HCO-VNLS'),
                         ['cutoff', 'model', 'features', 'accuracy', 'precision_risk', 'recall_risk',
                          'f1_risk', 'average_precision_risk', 'fit_seconds']].sort_values(['cutoff', 'model']).round(5))
else:
    print('Chưa có CatBoost mới; không thêm số liệu cũ khác cấu hình.')